# Milestone 2: ViT registers reproduction (run on Google Colab)

This notebook runs the whole pipeline for this milestone, top to bottom:

1. **Tests**: check that our model and measurement code behave correctly
2. **Data + forward pass check**: download Tiny-ImageNet, run one batch through our ViT (0 and 4 registers)
3. **Sanity training**: short runs on a small subset, to show the training loop learns
4. **Part A**: reproduce the paper's findings on Meta's pretrained DINOv2 models
5. **Download**: zip all results so they can be uploaded to GitHub

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Takes about 30–40 minutes.

## 0. Setup: get the code

In [ ]:
import os

REPO_URL = "https://github.com/YOUR-USERNAME/vit-registers-reproduction.git"  # <-- change to your repo

if os.path.exists("../src") and not os.path.exists("src"):  # opened from inside notebooks/
    %cd ..
if not os.path.exists("src"):                                  # fresh Colab session
    !git clone -q {REPO_URL} repo
    %cd repo
!pip install -q -r requirements.txt

import torch
from IPython.display import Image, Markdown, display
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime to T4!)")

## 1. Tests

In [ ]:
!python -m pytest -q

## 2. Data pipeline + forward pass check (Week 3 deliverable)

In [ ]:
!python -m scripts.check_forward_pass

In [ ]:
display(Markdown(open("results/logs/forward_pass_check.md").read()))
display(Image("results/figures/tinyimagenet_samples.png"))

## 3. Sanity training runs
ViT-Tiny, 10k training images, 5 epochs, 0 vs 4 registers. Loss should go down. These are not results, only a check that training works.

In [ ]:
!python -m src.train --registers 0 --epochs 5 --train-subset 10000 --val-subset 2000 --out results/runs/sanity_tiny_reg0
!python -m src.train --registers 4 --epochs 5 --train-subset 10000 --val-subset 2000 --out results/runs/sanity_tiny_reg4

In [ ]:
from src.plots import plot_training_curves

plot_training_curves(
    {"0 registers": "results/runs/sanity_tiny_reg0", "4 registers": "results/runs/sanity_tiny_reg4"},
    "results/figures/sanity_runs.png",
    "Sanity runs: ViT-Tiny, 10k-image subset of Tiny-ImageNet",
)
display(Image("results/figures/sanity_runs.png"))

## 4. Part A: artifacts in pretrained DINOv2 (no training)
Downloads 8 models (S, B, L, g, each with and without registers). If Colab runs out of memory on the giant model, change `S,B,L,g` to `S,B,L` and run the cell again.

In [ ]:
!python -m scripts.run_part_a --sizes S,B,L,g

In [ ]:
display(Markdown(open("results/part_a/summary.md").read()))
for name in ["norm_histograms", "outliers_by_size", "neighbour_cosine",
             "attention_maps", "norm_maps", "norm_histograms_after_layernorm"]:
    display(Markdown(f"**part_a_{name}**"))
    display(Image(f"results/figures/part_a_{name}.png"))

## 5. Save exact library versions and download everything

In [ ]:
!pip freeze | grep -iE "^(torch|torchvision|numpy|matplotlib|pytest)==" > requirements-lock.txt
!cat requirements-lock.txt
!rm -f results/runs/*/checkpoint.pt    # model weights are too big for GitHub; not needed yet
!zip -qr milestone2_results.zip results README.md requirements-lock.txt

try:
    from google.colab import files
    files.download("milestone2_results.zip")
except ImportError:
    print("Not on Colab: milestone2_results.zip is in", os.getcwd())